In [1]:
!pip install -q markdownify rank-bm25 sentence-transformers faiss-gpu-cu11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 60.1 MB/s eta 0:00:00


Сначала реализуем парсер статей с сайта. С помощью библиотек `BeautifulSoup`, `re` и `urllib` произведём рекурсивный обход всех подстраниц страницы `https://academy.lamoda.ru/articles` и сбор всех статей с последующим их переводом в .md формат. Будем дополнительно сохранять информацию о страницах и "граф страниц" в отдельных .json файлах для более быстрого доступа при повторных запросах к сайту (таким образом можно сохранять родительские и дочерние ссылки для каждой страницы). Отличать обычную промежуточную страницу от нужного нам материала будем по характерной фразе "Помогла эта информация?" (было замечено, что она есть в конце каждой статьи)

In [2]:
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md
import time
import re
import os
import sys
import json
from urllib.parse import urljoin

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; ColabParser/1.0)"}
BASE_URL = "https://academy.lamoda.ru"
OUTPUT_DIR = "./lamoda_articles"
INDEX_FILE = os.path.join(OUTPUT_DIR, "_index.json")
PAGE_GRAPH_FILE = os.path.join(OUTPUT_DIR, "_page_graph.json")
REQUEST_TIMEOUT = (3, 8)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
def load_json_file(path, default):
    if not os.path.exists(path):
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (OSError, json.JSONDecodeError):
        return default

def save_json_file(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

page_graph = load_json_file(PAGE_GRAPH_FILE, {})

def make_graph_node(url):
    node = page_graph.setdefault(url, {})
    node.setdefault("html", None)
    node.setdefault("is_article", None)
    node.setdefault("links", [])
    node.setdefault("found_from", [])
    return node

def save_page_graph():
    save_json_file(PAGE_GRAPH_FILE, page_graph)

def get_soup(url):
    node = make_graph_node(url)
    if node.get("html"):
        return BeautifulSoup(node["html"], "html.parser")

    resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    node["html"] = resp.text
    save_page_graph()
    return BeautifulSoup(resp.text, "html.parser")

def load_existing_articles():
    if os.path.exists(INDEX_FILE):
        try:
            with open(INDEX_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except (OSError, json.JSONDecodeError):
            pass

    existing = []
    for name in os.listdir(OUTPUT_DIR):
        if not name.endswith(".md"):
            continue
        filepath = os.path.join(OUTPUT_DIR, name)
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read(2000)
        except OSError:
            continue
        match = re.search(r"(?:Источник|Source):\s*(https?://\S+)", text)
        if not match:
            continue
        title = name.rsplit(".", 1)[0]
        title = re.sub(r"^\d+_", "", title)
        existing.append({"title": title, "url": match.group(1), "file": filepath})
    return existing

def parse_article(url):
    soup = get_soup(url)
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else "Без названия"
    content = soup.select_one(".article-detail__text") or soup.select_one(".article-detail-content")
    if not content:
        content = (
            soup.find("div", class_=re.compile("article-content|post-content", re.I))
            or soup.find("article")
        )
    if not content and not soup.select_one(".article-detail-outer"):
        body = soup.find("body")
        if body:
            for tag in body(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()
            content = body
        else:
            content = soup
    if not content:
        return title, ""
    content = BeautifulSoup(str(content), "html.parser")
    for tag in content(["script", "style", "nav", "footer", "header", "aside", "form", "button", "svg", "img"]):
        tag.decompose()
    for a in content.find_all("a"):
        a.unwrap()

    md_text = ""
    if content:
        md_text = md(str(content), heading_style="ATX", strip=["a", "img"])
    md_text = re.sub(r"\n{3,}", "\n\n", md_text).strip()
    return title, md_text

def save_index(articles_data):
    with open(INDEX_FILE, "w", encoding="utf-8") as f:
        json.dump(articles_data, f, ensure_ascii=False, indent=2)

def save_article(url, articles_data, processed_urls):
    if url in processed_urls:
        print(f"SKIP: {url}", flush=True)
        return False

    title, md_text = parse_article(url)
    if not md_text.strip():
        processed_urls.add(url)
        print(f"SKIP EMPTY: {url}", flush=True)
        return False
    safe_title = re.sub(r'[\\/*?:"<>|]', "", title)[:100]
    article_number = len(articles_data) + 1
    name = f"{article_number:03d}_{safe_title}.md" if safe_title else f"{article_number:03d}_article.md"
    filepath = os.path.join(OUTPUT_DIR, name)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        f.write(f"Источник: {url}\n\n")
        f.write(md_text)

    articles_data.append({"title": title, "url": url, "file": filepath})
    processed_urls.add(url)
    save_index(articles_data)
    print(f"OK: {title}", flush=True)
    return True

def normalize_url(url):
    return url.split("#")[0].split("?")[0]

def get_article_links(soup, page_url):
    links = set()
    for a in soup.find_all("a", href=True):
        full_url = normalize_url(urljoin(page_url, a["href"]))
        if full_url.startswith(BASE_URL + "/articles/"):
            links.add(full_url)
    return links

def is_that_article(soup):
    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    return "Помогла эта информация?" in text[-3000:]

def find_article_links(start_url, on_article=None):
    visited = set()  # чтобы не было повторений и для быстрого поиска элемента внутри
    article_urls = set()
    to_visit = [start_url]
    while to_visit:
        next_to_visit = []
        for url in to_visit:
            if url in visited:
                continue
            try:
                soup = get_soup(url)
                visited.add(url)
                article_links = get_article_links(soup, url)
                is_article = is_that_article(soup)
                node = make_graph_node(url)
                node["is_article"] = is_article
                node["links"] = sorted(article_links)
                for link in article_links:
                    child_node = make_graph_node(link)
                    found_from = set(child_node["found_from"])
                    found_from.add(url)
                    child_node["found_from"] = sorted(found_from)
                save_page_graph()
                if is_article:
                    article_urls.add(url)
                    if on_article:
                        on_article(url)
                next_to_visit.extend(article_links - visited)
                time.sleep(0.2)
            except Exception as e:
                print(f"Ошибка обхода {url}: {e}")
        to_visit = list(set(next_to_visit) - visited)
    return list(article_urls)

In [4]:
articles_data = load_existing_articles()
processed_urls = {item["url"] for item in articles_data}

def handle_article(url):
    try:
        save_article(url, articles_data, processed_urls)
    except Exception as e:
        print(f"FAIL {url}: {e}", flush=True)

article_urls = find_article_links(BASE_URL, on_article=handle_article)
print(f"Found {len(article_urls)} potential articles", flush=True)

for i, url in enumerate(article_urls):
    if url in processed_urls:
        print(f"[{i+1}/{len(article_urls)}] SKIP: {url}", flush=True)
        continue
    try:
        save_article(url, articles_data, processed_urls)
    except Exception as e:
        print(f"[{i+1}/{len(article_urls)}] FAIL {url}: {e}", flush=True)
    time.sleep(0.3)

save_index(articles_data)
print(f"Сохранено {len(articles_data)} статей", flush=True)

OK: Ошибки при подписании УПД с кодами маркировки: разбор кейсов
OK: Создание аккаунта в Lamoda Seller
OK: Утилизация товара FBO
OK: Отчет «Анализ корзины»
OK: Расхождения при приеме возврата FBS
OK: Инструкция по оформлению возвратов через Gate Management
OK: Performance-маркетинг
OK: Товары, запрещенные к продаже на Lamoda
OK: Копирование карточек товаров в другой личный кабинет
OK: О Lamoda API
OK: Остатки FBO в API
OK: Этикетки отгрузки в API
OK: Доверенность на возврат FBS
OK: Частые проблемы в API
OK: Упаковка товара для поставки FBO
OK: Заказы FBS
OK: Управление товарами в странах СНГ
OK: Закон о маркировке товаров в России
OK: Сертификация товаров
OK: Быстрый старт API
OK: OAuth 2.0 авторизация API
OK: Типы возвратов и статусы в API
OK: Первая поставка FBO
OK: Адреса складов для партнеров FBS
OK: «Подари вещам вторую жизнь» с фондом «Второе дыхание»
OK: Сравнение Lamoda Seller Partner API и Lamoda B2B Platform API
OK: Шаблон для обращений по API
OK: «Розовый месяц» с фондом «Да

#2. Ретривер
Теперь можно переходить к реализации ретривера. Для этого необходимо разбить текста на чанки - среди них и будет производиться поиск.         
Разбиение на чанки выберем простым - по фиксированному количеству слов с пересечением между чанками (для того, чтобы предложение, оборванное на середине предложение не теряло свой смысл и содержалось полностью хотя бы в одном чанке).         
Оно выбрано именно таким, т.к. сделать деление текста на чанки как-либо по-другому затруднительно, поскольку статьи имеют неодинаковую структуру. Например, у меня была идея поделить текст по фиксированному количеству предложений в чанке (для семантической связности), но, как я выяснил, некоторые статьи содержат внутри себя таблицы и куски кода, причём иногда большие. И как поделить такие элементы, соблюдая идею с предложениями - непонятно. Да и разбиение просто по словам с пересечением тоже неплохо сохраняет семантическую информацию.

In [5]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

chunk_size = 200  # запросы агента обычно состоят не более чем из 10 слов, поэтому такое количество слов будет не слишком большим, но и не слишком малым, чтобы не терялся контекст
chunk_overlap = 40  # вряд ли встретится предложение длиной более 40 слов

In [6]:
def load_articles_from_index(index_path, output_dir):
    with open(index_path, "r", encoding="utf-8") as f:
        articles_info = json.load(f)
    documents = []
    for art in articles_info:
        filepath = art["file"]
        url = art["url"]
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                md_text = f.read()
            body = re.split(r"\n\n", md_text, maxsplit=2)[2]
            documents.append({"text": body, "source": url})
        except OSError:
            continue
    return documents

def make_chunks(text, source, chunk_len=chunk_size, overlap=chunk_overlap):
    words = text.split()
    chunks = []
    idx = 0
    while idx < len(words):
        chunk_words = words[idx:idx+chunk_len]
        chunk_text = " ".join(chunk_words)
        if len(chunk_text) < chunk_overlap:  # если последний кусок слишком маленький, то отбрасываем его, так как он уже будет учтён в предыдущем чанке
            break
        chunks.append({"content": chunk_text, "source": source})
        idx += (chunk_len - overlap)
    return chunks

In [7]:
print("Загрузка статей и разбиение на чанки...")
articles_data = load_articles_from_index(INDEX_FILE, OUTPUT_DIR)
all_chunks = []
for doc in articles_data:
    all_chunks.extend(make_chunks(doc["text"], doc["source"]))
print(f"Получено {len(all_chunks)} чанков из {len(articles_data)} статей")

Загрузка статей и разбиение на чанки...
Получено 1277 чанков из 339 статей


Реализация sparse поиска. В качестве модели выбрана простая `Okapi BestMatch25`, потому что это достаточно хорошая модель, которая нормально ищет точные совпадения и не требует обучения (достаточно 1 раз токенизировать все документы, и потом модель будет быстро доставать нужные токены)

In [8]:
def tokenize(text):
    return re.findall(r'\w+', text.lower())  # в качестве токенизации просто разбиваем текст на слова и числа

corpus = [chunk["content"] for chunk in all_chunks]
tokenized_corpus = [tokenize(text) for text in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_search(query, top_k=20):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    topk_idx = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in topk_idx:
        if scores[idx] > 0:
            results.append({
                "content": all_chunks[idx]["content"],
                "source": all_chunks[idx]["source"],
                "score": scores[idx]
            })
    return results

Реализация dense поиска. В качестве модели выбрана уже обученная мультиязыковая модель `Multilingual E5 Large`. Она быстро работает внутри ноутбука (на GPU за минуты кодирует все нужные чанки), хорошо понимает русский и английский (что важно, ведь в статьях встречаются оба этих языка) и не требует дополнительного обучения.

In [9]:
model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
passages = [f"passage: {chunk['content']}" for chunk in all_chunks]
chunk_embs = model.encode(passages, normalize_embeddings=True, show_progress_bar=True)

dim = chunk_embs.shape[1]
res = faiss.StandardGpuResources()
index = faiss.GpuIndexFlatIP(res, dim)
index.add(chunk_embs.astype(np.float32))
print(f"Получили {index.ntotal} векторов")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Получили 1277 векторов


In [10]:
def dense_search(query, top_k=20):
    query_emb = model.encode([f"query: {query}"], normalize_embeddings=True)
    scores, indices = index.search(query_emb.astype(np.float32), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "content": all_chunks[idx]["content"],
                "source": all_chunks[idx]["source"],
                "score": float(score)
            })
    return results

Реализация смешанного ранжирования, которая учитывает результаты и sparse, и dense поиска. Для этого была создана функция `reciprocal_rank_fusion`, которая  оценивает итоговый `score` чанка иcходя из номеров его позиций в sparse и dense ретриверах. Формула для взвешенного score взята такой:  
$$RRFscore(d) = \frac{w_{sparse}}{k + rank_{sparse}(d)} + \frac{w_{dense}}{k + rank_{dense}(d)}$$ где:           

*   $k$ показывает, насколько большое преимущество имеют первые ранги по сравнению с остальными (чем меньше $k$, тем больше вклад первых)
*   $w_{sparse}$ и $w_{dense}$ показывают, насколько один `score` важнее, чем другой.             

Эта функция была выбрана именно такой, потому что выдаваемый ею `score` зависит лишь от рангов в sparse и dense ранжировании, но не от sparse и dense scores (а это хорошо, т.к. совсем не очевидно, как связаны sparse и dense scores между собой).

In [11]:
def reciprocal_rank_fusion(sparse_results, dense_results, sparse_weight=1, dense_weight=1, k=10):
    rrf_scores = {}
    doc_map = {}
    for rank, result in enumerate(sparse_results):
        doc_id = (result["source"], result["content"][:25] + result["content"][-25:])  # импровизированный ключ
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + sparse_weight / (k + rank + 1)
        doc_map[doc_id] = result
    for rank, result in enumerate(dense_results):
        doc_id = (result["source"], result["content"][:25] + result["content"][-25:])
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + dense_weight / (k + rank + 1)
        doc_map[doc_id] = result
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    top_docs = []
    for doc_id, rrf_score in sorted_docs:
        doc = doc_map[doc_id]
        top_docs.append({
            "content": doc["content"],
            "source": doc["source"],
            "rrf_score": rrf_score
        })
    return top_docs

def search_knowledge_base(query, top_k=5):
    sparse_result = sparse_search(query, top_k=20)
    dense_result = dense_search(query, top_k=20)
    mixed = reciprocal_rank_fusion(sparse_result, dense_result, sparse_weight=1, dense_weight=2)  # дадим dense search больший вес, т.к. так ответы становятся чуть лучше
    return mixed[:top_k]

In [12]:
test_query = "Как создать карточку товара?"
print(f"\nТестовый запрос: '{test_query}'")
results = search_knowledge_base(test_query, top_k=5)
for i, result in enumerate(results):
    print(f"{i+1}. {result['source']}")
    print(f"   {result['content']}")


Тестовый запрос: 'Как создать карточку товара?'
1. https://academy.lamoda.ru/articles/workingincis/kazakhstan/lokalnyy-marketpleys-kazakhstana/upravlenie-tovarami-v-kazakhstane/
   у вас немного товаров, вы можете создавать их по одному вручную. Для этого выберите «Добавить товар вручную» в разделе «Добавление товара». Выберите основную категорию и пол, а затем загрузите свои фото. На основе выбранных параметров формируется список атрибутов, которые нужно заполнить для создания товара. Затем товар отправится на модерацию. ### Добавление товара через Excel Выберите шаблон, скачайте и заполните все поля. ТН ВЭД можно выбрать в Справочнике. **! При создании товара через Excel нужно пользоваться справочниками, которые есть на листах Excel. Значения необходимо брать из справочников, название д****олжно соответствовать категории сайта (можно проверить на листе «Названия»). Затем по названию необходимо проверить ТН ВЭД.**
2. https://academy.lamoda.ru/articles/product-cards-and-photo-content/

In [13]:
test_query = "Как стать продавцом на Lamoda?"
print(f"\nТестовый запрос: '{test_query}'")
results = search_knowledge_base(test_query, top_k=5)
for i, result in enumerate(results):
    print(f"{i+1}. {result['source']}")
    print(f"   {result['content']}")


Тестовый запрос: 'Как стать продавцом на Lamoda?'
1. https://academy.lamoda.ru/articles/prodazha-so-sklada-lamoda-fbo/skhema-raboty-fbo/
   * Этикетки товаров оформлены по требованиям законодательства РФ * У вас есть сертификат качества или отказное письмо * Вы готовы обмениваться документами через электронный документооборот, используя формат УПД ## Как начать работу 1. Оставьте заявку на сайте Lamoda. Наш менеджер ответит вам через два дня или раньше. Он будет сопровождать вас на всех этапах и отвечать на вопросы, если они возникнут. 2. Заключите договор с Lamoda и пришлите на согласование ассортимент. 3. Загрузите карточки товаров в Lamoda Seller. Для карточки вам нужно сделать фотографии товаров и прислать их на согласование или воспользоваться нашей фотостудией. 4. Соберите поставку и отправьте товары в распределительный центр Lamoda. Подробнее о начале работы с Lamoda — в этом разделе.
2. https://academy.lamoda.ru/articles/start-working/nachalo-prodazh-na-marktepleyse-lamoda/
  

#3. ReAct цикл       
Реализация ReAct цикла агента (в качестве агента был выбран `YandexGPT`).   
Сначала определим функцию `agent_call`, которая будет возвращать текст ответа `YandexGPT` на отправленный ему запрос

In [15]:
import google.colab.userdata as userdata
YANDEX_API_KEY = userdata.get('YANDEX_API_KEY')
FOLDER_ID = userdata.get('FOLDER_ID')

In [16]:
def agent_call(messages, temperature=0.2):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Authorization": f"Api-Key {YANDEX_API_KEY}",
        "Content-Type": "application/json"
    }
    body = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": "2000"
        },
        "messages": messages
    }
    resp = requests.post(url, headers=headers, json=body)
    if resp.status_code != 200:
        raise Exception(f"YandexGPT error {resp.status_code}: {resp.text}")
    data = resp.json()
    return data["result"]["alternatives"][0]["message"]["text"]

Затем сформулируем системный промпт, чтобы агент выдавал ответы только в строго определённом формате

In [17]:
system_prompt = """Ты — эксперт по академии селлеров Lamoda. У тебя есть инструмент: search_knowledge_base(query: str) -> list[dict]
Он по твоему запросу query возвращает список релевантных статей с ключами 'content' и 'source' (URL).

Отвечай на вопросы пользователя, используя этот инструмент.
Формат ответа строгий:
Thought: Твои рассуждения
Action: search_knowledge_base(твой запрос)
ИЛИ
Final Answer: Окончательный ответ пользователю

После вызова инструмента тебе придет Search Result с результатами.
Если информации достаточно, дай Final Answer с ответом и ссылками на источники.
Если информации недостаточно, можешь ещё раз вызвать инструмент с уточнённым запросом, но не более пяти раз подряд. После пятого уточнения нужно обязательно выдать Final Answer

Формат Final Answer:
Final Answer: твой ответ на вопрос
Источники:
- [Название](https://...)

Пример:
Thought: Нужно узнать про создание карточки товара
Action: search_knowledge_base(создание карточки товара)
Observation: [{'content': '...', 'source': 'https://...'}, ...]
Thought: Информация найдена, можно отвечать
Final Answer: Для создания карточки товара...
Источники:
- [Название](https://...)
"""

Реализация функции, обрабатывающей последовательность запросов агенту, считывающей его ответы и обеспечивающей доступ агента к инструменту `search_knowledge_base`

In [18]:
def run_agent(user_query, max_steps=10):
    messages = [
        {"role": "system", "text": system_prompt},
        {"role": "user", "text": user_query}
    ]
    steps = 0
    while steps < max_steps:
        print(f"\nШаг {steps+1}")
        text = agent_call(messages, temperature=0.2)
        print(text)
        messages.append({"role": "assistant", "text": text})

        if "Final Answer:" in text:
            final = text.split("Final Answer:")[-1].strip()
            return final

        if "Action:" in text:
            action_line = [line for line in text.splitlines() if line.startswith("Action:")][0]
            match = re.search(r'search_knowledge_base\((.+?)\)', action_line)
            if match:
                query = match.group(1).strip().strip("'\"")
                results = search_knowledge_base(query, top_k=3)
                search_result = "Search Result:\n" + "\n".join(
                    [f"- {r['content']} (источник: {r['source']})" for r in results]
                )
                print(search_result)
                messages.append({"role": "user", "text": search_result})
            else:
                print("text")
                print("Нет запроса к search_knowledge_base, но и нет итогового ответа")
                messages.append({"role": "user", "text": "Формат твоего ответа не соответствует требованиям. У тебя нет запроса к search_knowledge_base, но и нет Final Answer. Необходимо следовать формату."})
                break
        else:
            messages.append({"role": "user", "text": "Формат твоего ответа не соответствует требованиям. Необходимо следовать формату."})
            print("FormatError")
        steps += 1
    return "Агент не смог дать ответ за отведённое количество шагов."

Запуск нескольких тестов с дополнительным выводом ответа агента и `Search Result` на каждом шагу (часть выводов уже реализована в функции выше, но их можно удалить при необходимости работы без вывода)

In [19]:
final_answer = run_agent("Какие документы нужны для регистрации продавца?")
print("\n\n\n\nИтоговый ответ:")
print(final_answer)


Шаг 1
Thought: Нужно узнать, какие документы требуются для регистрации продавца на Lamoda.
Action: search_knowledge_base(документы для регистрации продавца)
Search Result:
- Продажа оригинального товара — один из основных принципов работы Lamoda, поэтому мы всегда просим Партнеров предоставить документы, подтверждающие законность использования товарных знаков. Перед началом сотрудничества по новому бренду вы можете воспользоваться списком ниже для предварительной подготовки документов в зависимости от конкретной ситуации. **Если товарный знак зарегистрирован, принадлежит вам, то вы можете предоставить:** * Свидетельство на товарный знак **Если товарный знак зарегистрирован, но принадлежит третьему лицу:** * Документ, подтверждающий право использования товарного знака, например: разрешительное письмо правообладателя, лицензионный договор, дистрибьюторский договор или договор поставки **Если товарный знак находится в процессе регистрации:** * Заявку на регистрацию товарного знака с отме

In [20]:
final_answer = run_agent("Как работать с заказами через приложение Lamoda?")
print("\n\n\n\nИтоговый ответ:")
print(final_answer)


Шаг 1
Thought: Нужно узнать про работу с заказами через приложение Lamoda
Action: search_knowledge_base(работа с заказами через приложение Lamoda)
Search Result:
- FBS — это модель сотрудничества, при которой Lamoda не хранит товары на собственном складе, а получает, обрабатывает и доставляет уже готовые и упакованные заказы покупателям. ## Как устроена работа на FBS 1. Вы создаете товары и передаете доступный сток. 2. Lamoda размещает товары на своем сайте. 3. Клиент выбирает товары на сайте Lamoda. 4. Call-центр Lamoda подтверждает заказ. 5. Вы собираете заказ и готовите поставку для Lamoda. 6. Вы отправляете заказ на склад Lamoda. 7. Lamoda принимает поставку с заказами. 8. Доставка заказа клиенту Lamoda или сторонними службами. 9. Lamoda принимает товар, от которого отказался клиент. 10. Lamoda возвращает вам товары. 11. Lamoda передает вам актуальный статус заказа. 12. Вы передаете Lamoda статус возвращенного товара. **Преимущества:** * Вы храните товары на своем складе, Lamoda з

In [21]:
final_answer = run_agent("Как разместить рекламу своего бренда в Lamoda?")
print("\n\n\n\nИтоговый ответ:")
print(final_answer)


Шаг 1
Thought: Нужно узнать про размещение рекламы бренда в Lamoda
Action: search_knowledge_base(размещение рекламы бренда в Lamoda)
Search Result:
- **Продвижение бренда на Lamoda** Lamoda предлагает два удобных варианта продвижения: вы можете запускать рекламу самостоятельно в рекламном кабинете или воспользоваться экспертной поддержкой команды менеджеров для комплексных кампаний. ## Самостоятельное продвижение Для перформанса и быстрых продаж * Запускаете рекламу в кабинете Lamoda AD Platform. Для открытия рекламного кабинета необходимо заполнить форму. * Форматы: товарная и баннерная реклама * Полный контроль бюджетов и ставок * Бюджет: от 1 000 руб./день На Lamoda Ad Platform представлены 2 типа продвижения: 1. **Товарная реклама** позволяет увеличить видимость товаров и рост продаж прямо в каталоге Lamoda. 2. **Баннерная реклама** предоставляет возможность привлечь внимание пользователей к акциям, новинкам и коллекциям бренда. **В разделе товарного продвижения доступно промо тов